In [ ]:
torch_device = "cuda:1"

In [ ]:
# m is the LDM metadata
# e is the embedding
# r is the random bit
import json
import hmac
import hashlib
import pickle

m = {"model_id" : "stabilityai/stable-diffusion-2-1", "timesteps": 10}
def construct_POA(m, e, r, author_id = "1"):
    m = pickle.dumps(m)
    e = pickle.dumps(e)
    
    signature = hmac.new(
        bytes("1", 'latin-1'), 
        msg=pickle.dumps((e, m, r)), 
        digestmod=hashlib.sha256
    ).hexdigest().upper()

    return signature [:8]

In [ ]:
from datasets import load_dataset
train = [sample['Prompt'] for sample in load_dataset('Gustavosta/Stable-Diffusion-Prompts')['train']]

In [ ]:
from diffusers import DDPMPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler, DDIMScheduler
from PIL import Image
import torch
from tqdm.auto import tqdm

model_id = "stabilityai/stable-diffusion-2-1"
# batch_size=10

vae = AutoencoderKL.from_pretrained(model_id , subfolder="vae", use_safetensors=True)
tokenizer = CLIPTokenizer.from_pretrained(model_id , subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    model_id , subfolder="text_encoder", use_safetensors=True
)
unet = UNet2DConditionModel.from_pretrained(
    model_id , subfolder="unet", use_safetensors=True
)

vae.to(torch_device)
text_encoder.to(torch_device)
unet.to(torch_device)

prompt= ['carrots with coffee']
batch_size=len(prompt)

text_input = tokenizer(
    prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
)

with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]


max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt")
uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler", batch_size=batch_size)


In [ ]:
def convert_sentence_to_embed(lst_of_sentences):
    text_input = tokenizer(
        lst_of_sentences, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
    )
    batch_size = len(lst_of_sentences)
    with torch.no_grad():
        text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
    
    
    max_length = text_input.input_ids.shape[-1]
    uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt")
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
    return text_embeddings

import numpy as np
scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler", batch_size=batch_size)
    
def get_vae_embed_from_text_embed(text_embed, seed=1, timesteps=10, 
                                  n_samples=1, batch_size=batch_size,
                                 guidance_scale = 7.5,
                                 same_seed=False,
                                 latents=None): # use same seed across all iteration 
    
    
    out = []
    scheduler.set_timesteps(timesteps)
    out_latent = []
     #seed is a 64 bit signed int
    generator = torch.manual_seed(seed)
    for _ in tqdm([i for i in range(n_samples)]):
        if same_seed:
            generator = torch.manual_seed(seed)
        if latents is None:
            latents = torch.randn(
                (batch_size, unet.config.in_channels, 512//8, 512//8),
                    generator=generator,
                )
        latents = latents.to(torch_device)
        latents = latents * scheduler.init_noise_sigma
        
        for t in tqdm(scheduler.timesteps):
            
            # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
            latent_model_input = torch.cat([latents] * 2)
            latent_model_input = scheduler.scale_model_input(latent_model_input, timestep=t)

            # predict the noise residual
            with torch.no_grad():
                noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embed).sample

            # perform guidance
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

            # compute the previous noisy sample x_t -> x_t-1
            latents = scheduler.step(noise_pred, t, latents).prev_sample
        # vae step
        latents = 1 / 0.18215 * latents
        out_latent.append(latents)
    return torch.cat(out_latent,dim=0).to(torch_device)

def latent_to_img(latents):
    with torch.no_grad():
        image = vae.decode(latents).sample
    np_image = np.array(image.cpu())
#         if height % 8 != 0 or width % 8 != 0:

        # np_image = cv2.resize(np.transpose(np_image, (0,2, 3, 1)), dsize=dim_wh, interpolation=cv2.INTER_LINEAR)
    # np_image = np.transpose(np_image, (0,3,1,2))
    return np_image
    #     out.extend(np_image)
    # return np.array(out)

In [ ]:
# m is the LDM metadata
# e is the embedding
# r is the random bit
import json
import hmac
import hashlib
import pickle

m = {"model_id" : "stabilityai/stable-diffusion-2-1", "timesteps": 10}
def construct_POA(m, e, r, author_id = "1"):
    m = pickle.dumps(m)
    e = pickle.dumps(e)
    
    signature = hmac.new(
        bytes("1", 'latin-1'), 
        msg=pickle.dumps((e, m, r)), 
        digestmod=hashlib.sha256
    ).hexdigest().upper()

    return signature [:8]

In [ ]:
# ks_test_stat = []
from scipy.stats import *
p_val = []
n_prompts = 10
n_image = 30

def compute_pairwise_dist(arr):
    return [float(torch.sum(arr[i]*arr[j])) for i in range(len(arr)) for j in range(i+1, len(arr))]
    
for nn in range(n_prompts):
    # generate images
    print(nn)
    embeds_same_prompt_diff_seed = []
    prompt_embed = convert_sentence_to_embed(train[nn:nn+1])
    for i in range(n_image):
        if i == 0:
            seed = construct_POA(m=m, e=prompt_embed, r=1)
            seed = int(seed, base=16)
        else:
            seed = i
        latents = torch.randn(
                    (batch_size, unet.config.in_channels, 512//8, 512//8),
                        generator = torch.manual_seed(seed),
                    )
        # latents = torch.randn(
        #             (batch_size, unet.config.in_channels, 512//8, 512//8),
        #                 generator = torch.manual_seed(i),
        #             )
        # lats_same_prompt_diff_seed.append(latents)
        vae_embed = get_vae_embed_from_text_embed(prompt_embed, latents=latents)
        
        embeds_same_prompt_diff_seed.append(vae_embed)
    # compute ks
    same_prompt_pairwise_dist = compute_pairwise_dist(embeds_same_prompt_diff_seed)
    ratio_1 = [same_prompt_pairwise_dist[i]/4/64/64 for i in range(len(same_prompt_pairwise_dist))]
    # approx distr
    beta,loc,scale = gennorm.fit(ratio_1)

    # get image and invert back to latent
    im = latent_to_img(embeds_same_prompt_diff_seed[0])
    inv_vae_latent = vae.to(torch_device).encode(torch.from_numpy(im.astype(np.float32)).to(torch_device))

    T = torch.sum(embeds_same_prompt_diff_seed[0] * inv_vae_latent.latent_dist.mean)/4/64/64
    p_val.append(gennorm.sf(float(T), beta=beta, loc=loc, scale=scale))
    print(p_val[-1])

In [ ]:
loc

In [ ]:
np.std(ratio_1)

In [ ]:
T

In [ ]:
np.float64(2**-50)

In [ ]:
np.set_printoptions(precision=100)


print(gennorm.sf(float(T), beta=beta, loc=loc, scale=scale))

In [ ]:
torch.sum(embeds_same_prompt_diff_seed[0] * inv_vae_latent.latent_dist.mean)

In [ ]:
4*64*64

In [ ]:
inv_vae_latent.latent_dist.mean

In [ ]:
sb.histplot(ks_test_stat, bins=15, stat="density")